# 01 · Data harga dan audit gap

**Tujuan:** membaca data XAUUSD H1, memastikan datanya sehat, lalu membagi data menjadi *segmen* yang tidak terputus oleh gap tak wajar.

**Input:** `data/XAUUSD_H1_MetaQuotes_2020_2026.csv` (MetaQuotes-Demo, sama persis dengan input penelitian)
**Output:** `output/01_harga.parquet`

Padanan di kode penelitian: `protocol.py`, `gaps.py`, `market_audit.py`.

In [1]:
from pathlib import Path

import pandas as pd

DATA = Path("data")
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

## 1. Baca data

Setiap baris adalah satu bar H1. Kolom `time` adalah waktu **pembukaan** bar dalam UTC.

In [2]:
harga = pd.read_csv(DATA / "XAUUSD_H1_MetaQuotes_2020_2026.csv")
harga["time"] = pd.to_datetime(harga["time"], utc=True)
harga = harga[["time", "open", "high", "low", "close"]]

print("Jumlah bar :", len(harga))
print("Rentang    :", harga["time"].min(), "s.d.", harga["time"].max())
harga.head()

Jumlah bar : 38799
Rentang    : 2020-01-02 06:00:00+00:00 s.d. 2026-07-31 22:00:00+00:00


,time,open,high,low,close
0,2020-01-02 06:00:00+00:00,1520.26,1520.36,1519.39,1519.39
1,2020-01-02 07:00:00+00:00,1519.37,1520.36,1518.95,1520.19
2,2020-01-02 08:00:00+00:00,1520.20,1520.91,1519.30,1520.87
3,2020-01-02 09:00:00+00:00,1520.94,1521.37,1519.03,1519.11
4,2020-01-02 10:00:00+00:00,1519.11,1520.93,1518.38,1520.75


## 2. Pemeriksaan dasar

Waktu harus unik dan berurutan, harga positif, dan `high`/`low` konsisten dengan `open`/`close`. Tidak ada harga yang diisi atau diubah.

In [3]:
assert harga["time"].is_unique
assert harga["time"].is_monotonic_increasing
assert (harga[["open", "high", "low", "close"]] > 0).all().all()
assert (harga["high"] >= harga[["open", "close"]].max(axis=1)).all()
assert (harga["low"] <= harga[["open", "close"]].min(axis=1)).all()
print("Semua pemeriksaan dasar lulus")

Semua pemeriksaan dasar lulus


## 3. Gap: jarak antarbar lebih dari 1 jam

Pada data H1 yang sempurna, bar berikutnya selalu dibuka 1 jam kemudian. Kalau lebih dari 1 jam, ada **gap**. Kebanyakan gap itu wajar (jeda harian, akhir pekan, libur bursa).

In [4]:
jarak = harga["time"].diff()
ada_gap = jarak > pd.Timedelta(hours=1)

print("Jumlah gap:", ada_gap.sum())
jarak[ada_gap].value_counts().head(8)

Jumlah gap: 1705


time
0 days 02:00:00    1231
2 days 02:00:00     279
0 days 03:00:00      84
0 days 04:00:00      29
2 days 03:00:00      28
0 days 06:00:00      13
2 days 01:00:00      11
3 days 02:00:00      10
Name: count, dtype: int64

## 4. Gap tak terjelaskan → mulai segmen baru

Audit penelitian (`market_audit.py`) mencocokkan setiap gap dengan jadwal sesi MT5 dan kalender libur:

| Klasifikasi | Jumlah |
|---|---:|
| Jeda terjadwal (harian, akhir pekan) | 1.627 |
| Penutupan sesuai bursa (libur) | 69 |
| **Tak terjelaskan** | **9** |

Hanya 9 gap tak terjelaskan yang memutus data. Setelah gap seperti itu, indikator (RSI, MACD, σ24) harus dihitung ulang dari awal. Hasil audit itu kita tulis langsung di bawah: waktu bar **pertama setelah** gap.

In [5]:
GAP_TAK_TERJELASKAN = pd.to_datetime([
    "2021-06-15 15:00", "2021-06-18 18:00", "2021-09-24 21:00",
    "2025-01-07 07:00", "2025-07-03 12:00", "2025-07-04 10:00",
    "2025-10-23 22:00", "2026-02-27 03:00", "2026-03-12 00:00",
], utc=True)

# setiap waktu itu harus benar-benar bar setelah gap di data
assert all(ada_gap[harga["time"] == t].item() for t in GAP_TAK_TERJELASKAN)

harga["segment_id"] = harga["time"].isin(GAP_TAK_TERJELASKAN).cumsum()

harga.groupby("segment_id")["time"].agg(mulai="min", akhir="max", jumlah_bar="count")

,mulai,akhir,jumlah_bar
segment_id,,,
0,2020-01-02 06:00:00+00:00,2021-06-15 13:00:00+00:00,8550
1,2021-06-15 15:00:00+00:00,2021-06-18 16:00:00+00:00,71
2,2021-06-18 18:00:00+00:00,2021-09-24 18:00:00+00:00,1603
3,2021-09-24 21:00:00+00:00,2025-01-07 04:00:00+00:00,19402
4,2025-01-07 07:00:00+00:00,2025-07-03 03:00:00+00:00,2887
5,2025-07-03 12:00:00+00:00,2025-07-04 04:00:00+00:00,16
6,2025-07-04 10:00:00+00:00,2025-10-23 20:00:00+00:00,1822
7,2025-10-23 22:00:00+00:00,2026-02-26 21:00:00+00:00,2015
8,2026-02-27 03:00:00+00:00,2026-03-11 22:00:00+00:00,204


Hasilnya 10 segmen (9 reset). Simpan untuk notebook berikutnya.

In [6]:
harga.to_parquet(OUTPUT / "01_harga.parquet", index=False)
print("Tersimpan:", OUTPUT / "01_harga.parquet", harga.shape)

Tersimpan: output/01_harga.parquet (38799, 6)
